# CS 3110/5110: Data Privacy

## Homework 7


In [2]:
# Load the data and libraries
import pandas as pd
import numpy as np
import random
from scipy import stats
import matplotlib.pyplot as plt


def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)


def laplace_mech_vec(vec, sensitivity, epsilon):
    return [
        v + np.random.laplace(loc=0, scale=sensitivity / epsilon) for v in vec
    ]


def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(
        loc=0, scale=sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    )


def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [
        v
        + np.random.normal(
            loc=0,
            scale=sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon,
        )
        for v in vec
    ]


def pct_error(orig, priv):
    return np.abs(orig - priv) / orig * 100.0


adult = pd.read_csv(
    "https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv"
)
retail = pd.read_csv(
    "https://github.com/jnear/cs3110-data-privacy/raw/main/homework/retail.csv"
).dropna()

## Question 1 (10 points)

Implement the function `adult_hists`, which should return three histograms based on the `adult` dataset:

1. A 1-d histogram of the `Occupation` column
2. A 1-d histogram of the `Education` column
3. A 2-d histogram of the `Race` and `Sex` columns

Your function should have a **total** privacy cost of `epsilon` and a **person-level** unit of privacy.


In [3]:
def adult_hists(epsilon):
    hist_1 = adult.groupby("Occupation").size()
    hist_2 = adult.groupby("Education").size()
    hist_3 = adult.groupby(["Race", "Sex"]).size()

    hist_1_noisy = laplace_mech_vec(hist_1, sensitivity=1, epsilon=epsilon / 3)
    hist_2_noisy = laplace_mech_vec(hist_2, sensitivity=1, epsilon=epsilon / 3)
    hist_3_noisy = laplace_mech_vec(hist_3, sensitivity=1, epsilon=epsilon / 3)
    return (
        np.array(hist_1_noisy),
        np.array(hist_2_noisy),
        np.array(hist_3_noisy),
    )

In [4]:
# TEST CASE
occupation, education, multi = adult_hists(1.0)
assert occupation.sum() > 30000
assert education.sum() > 32000
assert multi.sum() > 32000

## Question 2 (10 points)

Argue that your implementation of `adult_hists` has a total privacy cost of `epsilon` and a person-level unit of privacy.


Each person contributes to exactly one bin in each histogram, so each histogram has an $L_1$ sensitivity of 1. The three histograms are computed from the same rows, so I use sequential composition rather than parallel composition: each Laplace mechanism uses $\epsilon/3$, making the total privacy cost $\epsilon/3 + \epsilon/3 + \epsilon/3 = \epsilon$. Since each person appears in one row, neighboring datasets differ by one person's row, so the guarantee is person-level.


## Question 3 (10 points)

Implement the function `retail_hist`, which should return one histogram based on the `retail` dataset:

1. A 1-d histogram of the `Country` column

Your function should have a **total** privacy cost of `epsilon` and an **event-level** unit of privacy.


In [5]:
def retail_hist(epsilon):
    hist = retail.groupby("Country").size()
    hist_noisy = laplace_mech_vec(hist, sensitivity=1, epsilon=epsilon)
    return np.array(hist_noisy)

In [6]:
# TEST CASE
country_hist = retail_hist(1.0)
assert country_hist.sum() > 7400

## Question 3 (10 points)

Argue that your implementation of `retail_hist` has a total privacy cost of `epsilon` and an event-level unit of privacy.


Each event is contained in exactly one row, and each row contributes to exactly one country bin, so the histogram has an $L_1$ sensitivity of 1. Adding Laplace noise with scale $1/\epsilon$ to each bin gives an $\epsilon$-differentially private vector release. Because neighboring datasets differ by one event, this guarantee is event-level.


## Question 4 (10 points)

Implement the function `retail_hist_bounded`, which returns a 1-d histogram of the `Country` column, with a total privacy cost of `epsilon`, and with a _user-level_ unit of privacy. Use contribution bounding to enforce the unit of privacy.


In [7]:
def retail_hist_bounded(epsilon):
    bounding_parameter = 200
    # organize the data by customerID
    retail_grouped = retail.groupby("CustomerID")
    # remove any excess purchases per customer
    retail_bounded = retail_grouped.head(bounding_parameter)

    # now using the bounded data, we do the normal histogram calculation
    hist = retail_bounded.groupby("Country").size()
    # ...making sure to use the bounding parameter as the new sensitivity
    hist_noisy = laplace_mech_vec(
        hist, sensitivity=bounding_parameter, epsilon=epsilon
    )
    return np.array(hist_noisy)

In [8]:
# TEST CASE
country_hist = retail_hist_bounded(1.0)
assert country_hist.sum() > 6000

## Question 5 (10 points)

Argue that your implementation of `retail_hist_bounded` has a total privacy cost of `epsilon` and a person-level unit of privacy.


The code keeps at most $k=200$ rows for each CustomerID, so one person's contribution can change at most 200 histogram counts in total. The histogram therefore has an $L_1$ sensitivity of 200, and adding Laplace noise with scale $200/\epsilon$ to each bin gives a total privacy cost of $\epsilon$. Assuming each CustomerID corresponds to one user, this contribution bound gives a user-level (person-level) guarantee.


## Question 6 (20 points)

Implement `retail_avg_quantity`, a function that returns the average quantity of orders in the `retail` dataset, with a total privacy cost of `epsilon` and a person-level unit of privacy.


In [57]:
def retail_avg_quantity(epsilon):
    # Parameters chosen by me
    bounding_parameter = 20
    clipping_parameter = 200

    # organize the data by customerID
    retail_grouped = retail.groupby("CustomerID")
    # remove any excess purchases per customer
    retail_bounded = retail_grouped.head(bounding_parameter)

    # clip the quantities to the max allowed
    clipped_quantity = retail_bounded["Quantity"].clip(
        lower=0, upper=clipping_parameter
    )
    sum_noisy = laplace_mech(
        clipped_quantity.sum(),
        sensitivity=bounding_parameter * clipping_parameter,
        epsilon=(epsilon * 3 / 4),
    )

    count_noisy = laplace_mech(
        len(retail_bounded),
        sensitivity=bounding_parameter,
        epsilon=(epsilon * 1 / 4),
    )

    return sum_noisy / count_noisy

In [67]:
# TEST CASE
avg = retail_avg_quantity(1)
assert avg > 7
assert avg < 13

## Question 7 (10 points)

Argue that your implementation of `retail_avg_quantity` has a total privacy cost of `epsilon` and a person-level unit of privacy.


Contribution bounding limits each user to 20 rows, and clipping each quantity to [0, 200] gives the sum query sensitivity $20 \cdot 200 = 4{,}000$; the count query has sensitivity 20. The noisy sum uses $3\epsilon/4$ and the noisy count uses $\epsilon/4$, so sequential composition gives a total privacy cost of $\epsilon$. The returned ratio is post-processing of these two private values and does not add privacy cost.


## Question 8 (10 points)

In 1-3 sentences each, answer the following:

- What impact did the choice of contribution bound have on the average?
- How did you arrive at your choice of contribution bound?
- What is the better choice of privacy unit for the `retail` dataset: event-level or person-level?
- What is an example of a potential negative outcome associated with choosing the event-level unit of privacy for this dataset?


- The bound of 20 limits each customer to at most 20 rows, so the average is
  calculated from the bounded data rather than all events. This can change how
  representative the average is, especially for customers with more than 20
  events, and the added noise introduces further error.
- I chose 20 as a tradeoff between limiting each customer's influence and
  retaining enough rows. I arrived at this value by inspecting the dataset,
  which would incur additional privacy cost in practice; a real release should
  choose the bound using public knowledge or account for a private selection
  procedure.
- Person-level is the better choice here because it protects all of a user's
  events together. Although this requires more noise than event-level privacy,
  it gives stronger protection against inferences about a user's participation.
- With event-level privacy, an adversary could use a person's repeated
  transactions to infer that they are in the dataset or learn behavioral
  patterns about them. The privacy guarantee would protect each individual
  event, but not the person's full transaction history.
